# Molab Orchestrator
> **Run these cells to automatically execute all notebooks sequentially and sync them to GitHub.**

In [ ]:
# Install dependencies
!pip install -q papermill tabulate

import os
import glob
import subprocess
import papermill as pm
from tqdm.notebook import tqdm

print('Dependencies loaded.')


In [ ]:
# Find all notebooks
all_notebooks = glob.glob('**/*.ipynb', recursive=True)
all_notebooks = [nb for nb in all_notebooks if 'Master_Runner' not in nb 
                 and 'molab_run' not in nb 
                 and '.ipynb_checkpoints' not in nb]
all_notebooks.sort()

print(f'Found {len(all_notebooks)} notebooks to execute sequentially.')


In [ ]:
# Sequential execution and GitHub push
results = []
for nb_path in tqdm(all_notebooks, desc="Running Notebooks"):
    print(f'\n--- STARTING: {nb_path} ---')
    try:
        pm.execute_notebook(
            input_path=nb_path,
            output_path=nb_path,
            cwd='.',
            log_output=True
        )
        results.append(nb_path)
        print(f'--- FINISHED: {nb_path} ---')
        
        # Commit and push immediately after successful completion
        subprocess.run(['git', 'add', nb_path])
        subprocess.run(['git', 'commit', '-m', f'Auto-update {nb_path} results'])
        subprocess.run(['git', 'push'])
        print(f'--- PUSHED {nb_path} to GitHub ---')
        
    except Exception as e:
        print(f'\n[ERROR] Failed to run {nb_path}: {e}')

print(f'\n✅ Execution Complete! Successfully ran and pushed {len(results)} out of {len(all_notebooks)} notebooks.')
